# Criação do Dataset Balanceado

## CICDDoS2019

### Dia de coleta 01/12

In [ ]:
from src.Data.Handler import DatasetHandler

handler = DatasetHandler(logging=True)

handler.create_balanced_dataset(
    src_dir="datasets/CICDDoS2019/Origin/01-12", 
    dest_dir="datasets/CICDDoS2019/data/01-12/Classification", 
    output_filename="BALANCED.CSV", 
    n_samples_per_class=10000,
    chunk_size=100000,
    target_files=None,
    ignored_classes=['WebDDoS'],
    allow_insufficient=True
)

# Análise de Similaridade

## CICDDoS2019

### Dia de coleta 01/12

In [ ]:
import pandas as pd
from src.Data.Processor import DataStreamProcessor
from src.Data.Handler import DatasetHandler

# Instancia as duas ferramentas
processor = DataStreamProcessor(logging=True)
handler = DatasetHandler(logging=True)

# Carrega o CSV 
df_bruto = pd.read_csv('datasets/CICDDoS2019/data/01-12/BALANCED.CSV')

# Pré-Processamento 
X_df, y_array, target_names = processor.create_stream(
    df=df_bruto,
    target_label_col='Label',
    binary_label=False,               # Falso pois queremos ver CADA ataque separado
    normalize_method='MinMaxScaler',  # Normaliza entre 0 e 1
    imputation_method='0',            # Substitui NaNs por 0
    extra_ignore_cols=['Source IP', 'Source Port','Destination IP','Destination Port','Protocol', 'Inbound'],
    return_stream=False               # Retorna (DataFrame de Features, Array de Labels, Nomes)
)

# Executa a Seleção OvR (One-vs-Rest)
dict_features, vetor_features_finais = handler.extract_ovr_feature_importance(
    X=X_df, 
    y=y_array, 
    target_names=target_names,
    top_per_class=7 
)

# Plota os Dendrogramas e Mapas de Calor
handler.plot_similarity_and_feature_groups(
    X=X_df, 
    y=y_array, 
    target_names=target_names,
    selected_features=vetor_features_finais, 
    metric='euclidian',   # cosine / euclidian
    linkage_method='ward' # average / ward 
)

# Criando gráfico de radar

### Coleta dia de coleta 01/12

In [ ]:
from src.Data.Processor import DataStreamProcessor
from src.Data.Handler import DatasetHandler
import pandas as pd

handler = DatasetHandler(logging=True)
processor = DataStreamProcessor(logging=True)

df = pd.read_csv('datasets/CICDDoS2019/data/01-12/BALANCED.CSV')

stream, targets, features = processor.create_stream(
    df=df, 
    target_label_col='Label', 
    binary_label=False, 
    normalize_method="MinMaxScaler", 
    threshold_var=0.001,
    threshold_corr=None,
    top_n_features=15,
    return_stream=False,
    extra_ignore_cols=['Source IP', 'Source Port','Destination IP','Destination Port','Protocol'],
    imputation_method='mediana'
)

handler.plot_feature_radar(stream, targets, features)
handler.plot_mini_radars(stream, targets, features)

# Criação de cenários CICDDoS2019 01/12

## Amostras: 15k 

In [ ]:
AMOSTRAS = [25, 200, 1000]
BASELINE = "data/BENIGN.csv"
INPUT_FOLDER = "datasets/CICDDoS2019/01-12"
OUTPUT_FOLDER = "data/15k"

### Cenário de Consistência

![CONSISTENCIA](datasets/CICDDoS2019/img/CONSISTENCIA.png)

In [ ]:
from src.Data.ScenarioGenerator import ScenarioGenerator

for qtd_amostras in AMOSTRAS:
    
    meu_cenario = [
        ("DrDoS_DNS.csv", qtd_amostras),
        ("DrDoS_DNS.csv", qtd_amostras),
        ("DrDoS_DNS.csv", qtd_amostras)
    ]

    caminho_saida = f"{OUTPUT_FOLDER}/Consistência/Consistência_{qtd_amostras}.csv"

    gerador = ScenarioGenerator(
        input_folder=INPUT_FOLDER,
        output_path=caminho_saida,
        baseline_file=BASELINE,
        logging=True,
        n_benign_samples=15000, 
        sort_by_timestamp=False,
        remove_duplicates=False
    )

    gerador.generate(meu_cenario)
    gerador.plot_scenario(window_size=250, features_plot=['Total Fwd Packets', 'Total Backward Packets'])

### Cenário Generalização

![GENERALIZACAO](datasets/CICDDoS2019/img/GENERALIZACAO.png)

In [ ]:
from src.Data.ScenarioGenerator import ScenarioGenerator

for qtd_amostras in AMOSTRAS:
    
    meu_cenario = [
        ("DrDoS_DNS.csv", qtd_amostras),
        ("DrDoS_LDAP.csv", qtd_amostras),
        ("DrDoS_DNS.csv", qtd_amostras)
    ]

    caminho_saida = f"{OUTPUT_FOLDER}/Generalização/Generalização_{qtd_amostras}.csv"

    gerador = ScenarioGenerator(
        input_folder=INPUT_FOLDER,
        output_path=caminho_saida,
        baseline_file=BASELINE,
        logging=True,
        n_benign_samples=15000, 
        sort_by_timestamp=False,
        remove_duplicates=False
    )

    gerador.generate(meu_cenario)
    gerador.plot_scenario(window_size=250, features_plot=['Total Fwd Packets', 'Total Backward Packets'])

### Cenário Adaptação

![ADAPTACAO](datasets/CICDDoS2019/img/ADAPTACAO.png)

In [ ]:
from src.Data.ScenarioGenerator import ScenarioGenerator

for qtd_amostras in AMOSTRAS:
    
    meu_cenario = [
        ("DrDoS_DNS.csv", qtd_amostras),
        ("Syn.csv", qtd_amostras),
        ("DrDoS_DNS.csv", qtd_amostras)
    ]

    caminho_saida = f"{OUTPUT_FOLDER}/Adaptação/Adaptação_{qtd_amostras}.csv"

    gerador = ScenarioGenerator(
        input_folder=INPUT_FOLDER,
        output_path=caminho_saida,
        baseline_file=BASELINE,
        logging=True,
        n_benign_samples=15000, 
        sort_by_timestamp=False,
        remove_duplicates=False
    )

    gerador.generate(meu_cenario)
    gerador.plot_scenario(window_size=250, features_plot=['Total Fwd Packets', 'Total Backward Packets'])

### Cenário de Recorrência

![RECORRENCIA](datasets/CICDDoS2019/img/RECORRENCIA.png)

In [ ]:
from src.Data.ScenarioGenerator import ScenarioGenerator

for qtd_amostras in AMOSTRAS:
    
    meu_cenario = [
        ("DrDoS_DNS.csv", qtd_amostras//2),
        ("Syn.csv", qtd_amostras//2),
        ("DrDoS_LDAP.csv", qtd_amostras//2),
        ("DrDoS_DNS.csv", qtd_amostras//2),
        ("Syn.csv", qtd_amostras//2),
        ("DrDoS_LDAP.csv", qtd_amostras//2)
    ]

    caminho_saida = f"{OUTPUT_FOLDER}/Recorrência/Recorrência_{qtd_amostras}.csv"

    gerador = ScenarioGenerator(
        input_folder=INPUT_FOLDER,
        output_path=caminho_saida,
        baseline_file=BASELINE,
        logging=True,
        n_benign_samples=15000, 
        sort_by_timestamp=False,
        remove_duplicates=False
    )

    gerador.generate(meu_cenario)
    gerador.plot_scenario(window_size=250, features_plot=['Total Fwd Packets', 'Total Backward Packets'])

## Base de Dados para otimização optuna

In [ ]:
from src.Data.ScenarioGenerator import ScenarioGenerator
    
meu_cenario = [
    ("DrDoS_DNS.csv", 250),
    ("Syn.csv", 1000),
    ("DrDoS_LDAP.csv", 50),
    ("DrDoS_DNS.csv", 500),
    ("Syn.csv", 2000),
    ("DrDoS_LDAP.csv", 25)
]

caminho_saida = f"{OUTPUT_FOLDER}/Otimização/Otimização.csv"

gerador = ScenarioGenerator(
    input_folder=INPUT_FOLDER,
    output_path=caminho_saida,
    baseline_file=BASELINE,
    logging=True,
    n_benign_samples=15000, 
    sort_by_timestamp=False,
    remove_duplicates=False
)

gerador.generate(meu_cenario)
gerador.plot_scenario(window_size=250, features_plot=['Total Fwd Packets', 'Total Backward Packets'])